# Exp 35: Training curves of submissions

Card: [exp35_training_curves.md](../../docs/experiments/exp35_training_curves.md)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from core.training.config import ROOT
from core.training.cv import CVResult
from core.analysis.plots import save_figures, use_style

FOLDS = [0, 1, 2, 3, 4]
CONFIG_DIR = ROOT / "configs" / "models"
MODELS = {
    "exp11": CONFIG_DIR / "exp11_resnet18_5fold" / "resnet18.yaml",
    "exp26": CONFIG_DIR / "exp26_coarse_dropout_5fold" / "resnet18.yaml",
    "exp29": CONFIG_DIR / "exp29_downsample_5fold" / "resnet18.yaml",
}
FIGURES = ROOT / "docs" / "figures"
PREFIX = "exp35_training_curves"

# dataviz reference palette, light mode
SERIES = {"train": "#eb6834", "validation": "#2a78d6"}
use_style()

## Per-epoch history

In [ ]:
curves = pd.concat(
    {name: CVResult.load(c).history().query("fold in @FOLDS") for name, c in MODELS.items()}, names=["model"]
).reset_index(level=0)


## Loss and mAP curves

In [ ]:
figures = {}
for metric in ["loss", "mAP"]:
    fig, axes = plt.subplots(1, len(MODELS), figsize=(4.2 * len(MODELS), 3.6), sharey=True)
    for name, ax in zip(MODELS, axes):
        part = curves[(curves["model"] == name) & (curves["metric"] == metric)]
        for (split, fold), line in part.groupby(["split", "fold"]):
            ax.plot(line["epoch"], line["value"], color=SERIES[split], lw=1, alpha=0.7, label=split if fold == FOLDS[0] else None)
        ax.set(title=name, xlabel="epoch", xticks=range(1, int(part["epoch"].max()) + 1))
    axes[0].set_ylabel(metric)
    axes[0].legend(loc="best")
    fig.suptitle(f"{metric} per epoch, one line per fold", color="#52514e")
    fig.tight_layout()
    figures[f"{PREFIX}_{metric.lower()}.png"] = fig
plt.show()

In [ ]:
save_figures(figures, FIGURES)